# Model pobreza

Model pobreza

## 1. Carga y preparación de datos

- Carga del dataset y renombramiento de columnas a nombres descriptivos
- Limpieza de tipos numéricos con manejo de nulos y totales residuales

In [161]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from pathlib import Path
from scipy import stats
from statsmodels.stats.multitest import fdrcorrection

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

cwd = Path.cwd().resolve()
root = cwd.parent if cwd.name == 'src' else cwd
DATA_DIR = root / 'data'
DOC_DIR = root / 'doc'
EXPORT_DIR = DATA_DIR / 'export'


In [162]:
df = pd.read_csv(DATA_DIR / 'INDICADORES_ESTUDIO_GRAL_V1.csv')

rename_map = {
    'IND 1.1': 'WEI',
    'IND 1.2': 'TCMA_URBANO',
    'CAGR_AGRICULTURA': 'TCMA_AGRICULTURA',
    'CAGR_AGUA': 'TCMA_AGUA',
    'CAGR_BOSQUE': 'TCMA_BOSQUE',
    'IND 1.3': 'Porc_poblacion_en_urbana',
    'IND 1.4': 'Densidad_urbana',
    'IND 2.1': 'Ind_POT',
    'IND 2.2': 'Ind_eficiencia_recaudo',
    'IND 2.3': 'Ingresos_tributarios_percap',
    'IND 2.4': 'Percepcion_corrupcion',
    'IND 2.5': 'Percepcion_verde',
    'IND 3.1': 'Porc_pobreza_monetaria',
    'IND 3.2': 'Porc_pobreza_extrema',
    'IND 3.3': 'Porc_pobreza_multi',
    'IND 3.4': 'GINI',
    'IND 3.5': 'Porc_informal',
    'IND 4.1': 'Porc_deficit_vivienda_cuali',
    'IND 4.2': 'Porc_deficit_vivienda_cuanti',
    'IND 4.3': 'Porc_contaminacion_aire',
    'IND 4.4': 'Porc_contaminacion_agua',
}
df = df.rename(columns=rename_map)
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print()
df.head(25).to_string()

Shape: (25, 33)
Columns: ['SUBREGIÓN', 'MUNICIPIO', 'AREA KM2', 'AREA KM2 URBANA', 'POBLACION ESTIMADA 2021/RURAL', 'POBLACION ESTIMADA 2021/URBANA', 'TOTAL POBLACIÓN', 'WEI', 'TCMA_URBANO', 'Porc_poblacion_en_urbana', 'Densidad_urbana', 'IND 2.1 TIPO', 'Ind_POT', 'AÑOS REVISION', 'AÑO ULTIMA REVISION', 'IND 2.1 REVISION', 'IND 2.1 ZONAS AMB', 'Ind_eficiencia_recaudo', 'Ingresos_tributarios_percap', 'Percepcion_corrupcion', 'Percepcion_verde', 'Porc_pobreza_monetaria', 'Porc_pobreza_extrema', 'Porc_pobreza_multi', 'GINI', 'Porc_informal', 'Porc_deficit_vivienda_cuali', 'Porc_deficit_vivienda_cuanti', 'Porc_contaminacion_aire', 'Porc_contaminacion_agua', 'TCMA_AGRICULTURA', 'TCMA_BOSQUE', 'TCMA_AGUA']



'              SUBREGIÓN     MUNICIPIO     AREA KM2 AREA KM2 URBANA  POBLACION ESTIMADA 2021/RURAL  POBLACION ESTIMADA 2021/URBANA  TOTAL POBLACIÓN     WEI TCMA_URBANO Porc_poblacion_en_urbana Densidad_urbana IND 2.1 TIPO Ind_POT AÑOS REVISION  AÑO ULTIMA REVISION  IND 2.1 REVISION IND 2.1 ZONAS AMB Ind_eficiencia_recaudo Ingresos_tributarios_percap Percepcion_corrupcion Percepcion_verde Porc_pobreza_monetaria Porc_pobreza_extrema Porc_pobreza_multi    GINI Porc_informal Porc_deficit_vivienda_cuali Porc_deficit_vivienda_cuanti Porc_contaminacion_aire Porc_contaminacion_agua TCMA_AGRICULTURA TCMA_BOSQUE TCMA_AGUA\n0   Sabana noroccidente      El Rosal     87,12009           5,403                         2494.0                         22051.0          24545.0  0,1494      1,7158                   88,40%     4081,251157          EOT    0,75         -6,00               2015.0               1.0              0,50            62,04/47,12                     367.868                33,24%       

In [163]:
print(df.dtypes)
print()
print('Rows:', len(df))
print('SUBREGIÓN values:', df['SUBREGIÓN'].unique())

SUBREGIÓN                          object
MUNICIPIO                          object
AREA KM2                           object
AREA KM2 URBANA                    object
POBLACION ESTIMADA 2021/RURAL     float64
POBLACION ESTIMADA 2021/URBANA    float64
TOTAL POBLACIÓN                   float64
WEI                                object
TCMA_URBANO                        object
Porc_poblacion_en_urbana           object
Densidad_urbana                    object
IND 2.1 TIPO                       object
Ind_POT                            object
AÑOS REVISION                      object
AÑO ULTIMA REVISION               float64
IND 2.1 REVISION                  float64
IND 2.1 ZONAS AMB                  object
Ind_eficiencia_recaudo             object
Ingresos_tributarios_percap        object
Percepcion_corrupcion              object
Percepcion_verde                   object
Porc_pobreza_monetaria             object
Porc_pobreza_extrema               object
Porc_pobreza_multi                

In [164]:
df.groupby('SUBREGIÓN')['MUNICIPIO'].unique().to_dict()

{'Guavio': array(['La Calera'], dtype=object),
 'Sabana Centro': array(['Cajicá ', 'Chía ', 'Cota', 'Sopó ', 'Tabio ', 'Tenjo'],
       dtype=object),
 'Sabana Norte': array(['Gachancipá  ', 'Tocancipá  ', 'Zipaquirá'], dtype=object),
 'Sabana noroccidente': array(['El Rosal', 'Facatativá ', 'Subachoque'], dtype=object),
 'Sabana suroccidente': array(['Bojacá  ', 'Funza ', 'Madrid ', 'Mosquera', 'Zipacón'],
       dtype=object),
 'Soacha - Sibaté': array(['Sibaté', 'Soacha'], dtype=object),
 'TOTALES': array([nan], dtype=object)}

In [165]:
def clean_numeric(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    s = s.replace('%', '')
    s = s.replace('.', '')
    if '/' in s:
        s = s.split('/')[0]
    s = s.replace(',', '.')
    try:
        return float(s)
    except ValueError:
        return np.nan
# 'TCMA_AGRICULTURA', 'TCMA_AGUA', 'TCMA_BOSQUE', 'WEI', 
indep_vars = ['TCMA_URBANO', 'Ind_POT', 'Porc_contaminacion_aire', 'Porc_deficit_vivienda_cuanti', 'SUBREGIÓN'] #'Porc_informal', 'Porc_contaminacion_agua', 'Densidad_urbana', 
dep_vars = ['Porc_pobreza_monetaria']

continuous_vars = [v for v in indep_vars if v != "SUBREGIÓN"]
all_index_cols = continuous_vars + dep_vars

for col in all_index_cols:
    df[col] = df[col].apply(clean_numeric)

df_num = df.dropna(subset=['MUNICIPIO'] + all_index_cols).copy()
df_num = df_num[~df_num['MUNICIPIO'].str.contains('TOTALES', case=False, na=False)]
df_num = df_num[~df_num['MUNICIPIO'].str.strip().isin(['', 'nan'])]

print(f'Rows after cleaning: {len(df_num)}')
print(f'Independent cols (numeric): {df_num[continuous_vars].dtypes.to_dict()}')

Rows after cleaning: 20
Independent cols (numeric): {'TCMA_URBANO': dtype('float64'), 'Ind_POT': dtype('float64'), 'Porc_contaminacion_aire': dtype('float64'), 'Porc_deficit_vivienda_cuanti': dtype('float64')}


In [166]:
df_num[indep_vars].describe()

,TCMA_URBANO,Ind_POT,Porc_contaminacion_aire,Porc_deficit_vivienda_cuanti
count,20.000000,20.000000,20.000000,20.00000
mean,4.776730,0.825000,12.509500,1.48650
std,2.726775,0.230845,8.965934,0.85747
min,1.715800,0.500000,3.030000,0.25000
25%,3.055125,0.500000,6.065000,0.66500
50%,4.333200,1.000000,9.855000,1.41000
75%,5.128400,1.000000,14.465000,2.15500
max,14.589000,1.000000,34.620000,3.37000


## 2. Model pobreza

- Selección de variables para usar

In [167]:
# Separate continuous and categorical independent variables
continuous_vars = [v for v in indep_vars if v != 'SUBREGIÓN']

# # Standardize continuous independent variables
# scaler = StandardScaler()
# df_num[continuous_vars] = scaler.fit_transform(df_num[continuous_vars])

# Build design matrix
X = df_num[continuous_vars]
y = pd.to_numeric(df_num[dep_vars[0]], errors='coerce')

# Ensure all columns are numeric and drop rows with missing values
X = X.apply(pd.to_numeric, errors='coerce')
model_data = pd.concat([X, y], axis=1).dropna()
X_clean = model_data.drop(columns=[dep_vars[0]])
y_clean = model_data[dep_vars[0]].values

X_clean = sm.add_constant(X_clean)

# Fit OLS regression
model = sm.OLS(y_clean, np.asarray(X_clean)).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.379
Model:                            OLS   Adj. R-squared:                  0.213
Method:                 Least Squares   F-statistic:                     2.288
Date:                Wed, 29 Jul 2026   Prob (F-statistic):              0.108
Time:                        23:18:43   Log-Likelihood:                -61.164
No. Observations:                  20   AIC:                             132.3
Df Residuals:                      15   BIC:                             137.3
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         19.5214      7.280      2.682      0.0

In [168]:
pd.DataFrame(zip(continuous_vars, model.params[1:]),columns=['Variable', 'Coeficiente'])

,Variable,Coeficiente
0,TCMA_URBANO,0.275821
1,Ind_POT,-9.344229
2,Porc_contaminacion_aire,-0.005686
3,Porc_deficit_vivienda_cuanti,3.905633
